## Get all the imports

In [1]:
import pytorch_lightning as pl
import torch
from torchvision import transforms
from torchvision.datasets import CIFAR10
from model import MyModel
from datacull.data import DCDataset
from datacull.methods.RCAP import RCAPDataLoader, RCAPImportance
from datacull.logger import DCLogger

## Defining some global variables

In [2]:
num_epochs = 20
batch_size = 256
pruning_rate = 0.8
beta = 1/4

## Create the data module class (CIFAR10 for simplicity)
- Since RCAP is a dynamic pruning technique, we need to create the data module only once

In [3]:
class DataModule(pl.LightningDataModule):
    def __init__(self, batch_size, sample_importance_object, pruning_rate):
        self.batch_size = batch_size
        self.pruning_rate = 1 - pruning_rate
        # This is the RCAPImportance object
        # This will be called at the beginning of each epoch to 
        # re-compute sample importances
        self.sample_importance_object = sample_importance_object
        # These are the transforms to be used if you want to train a model from scratch on the current dataset
        self.train_transform = transforms.Compose([
            transforms.RandomCrop(32, 4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ])
        self.val_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
        ])
        super().__init__()
    
    
    def prepare_data(self, stage="None"):
        # This is the only line where we differ from normal code.
        # We wrap the CIFAR10 dataset (or your own dataset) with DPDataset to enable indexing
        self.train_set = DCDataset(CIFAR10(root='data/', train=True, download=True, transform=self.train_transform))
        self.val_set = DCDataset(CIFAR10(root='data/', train=False, download=True, transform=self.val_transform))
    
    
    def setup(self, stage: str):
        self.train_data_loader = RCAPDataLoader(self.train_set, pruning_rate=self.pruning_rate, batch_size=self.batch_size, shuffle=True, num_workers=1)
        self.val_data_loader = torch.utils.data.DataLoader(self.val_set, batch_size=self.batch_size, shuffle=False, num_workers=1)
    
    
    def train_dataloader(self):
        # Resample the training data loader based on sample importance
        self.train_data_loader.resample(self.sample_importance.compute_importance())
        return self.train_data_loader
    
    
    def val_dataloader(self):
        return self.val_data_loader

## Define the logger object
- This object logs a model's metrics such as predictions, loss, etc.

In [4]:
logger_object = DCLogger(trajectory_dir="model_trajectory_loss/", save_every_k_epoch=1)

## Define the importance criteria
- Since RCAP is a dynamic pruning algorithm, its importance criteria needs to be computed every epoch
- Hence, the .compute_importance() needs to be called every epoch

In [5]:
data_module = DataModule(sample_importance_object=None, pruning_rate=pruning_rate, batch_size=batch_size)
data_module.prepare_data()
importance_object = RCAPImportance(dataset=data_module.train_set, logger_object=logger_object, beta=beta)
data_module.sample_importance = importance_object

## Define a model for training
- The logger object is used to log the metrics every iteration
- Check line 41 in model.py

In [6]:
model = MyModel(logger_object=logger_object)

## Create the data module object and train the model

In [7]:
# reload_dataloaders_every_n_epochs=True is important to ensure that the dataloader is reloaded every epoch to reflect the new sampling.
# For static methods, this only shuffles the data once the subset is selected, but for dynamic methods, this is necessary to update the sampling based on new importance scores.
# This is a design choice to unify both static and dynamic methods under the same flag.
trainer = pl.Trainer(accelerator="gpu", devices=1, max_epochs=num_epochs, deterministic=True, enable_model_summary=True, num_sanity_val_steps=0, reload_dataloaders_every_n_epochs=True)
trainer.fit(model, data_module)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
c:\ProgramData\anaconda3\Lib\site-packages\pytorch_lightning\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type               | Params | Mode 
---------------------------------------------------------

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.
